# TFM — 02. Preprocessing: Accidentes de tráfico en Madrid (2012-2018)
**Autora:** Meritxell Abellan Collado

Este notebook parte de las conclusiones del EDA (`01_EDA.ipynb`) y construye el dataset limpio y estructurado, a **nivel de accidente**, que servirá de entrada al notebook de *feature engineering*.

## Estructura

0. Carga de librerías y datos crudos
1. Limpieza estructural (nivel persona)
2. Construcción del target y agregación persona → accidente
3. Tipado y consistencia de variables
4. Missing data
5. Variable numérica y outliers
6. Train / test split
7. Control de calidad final y guardado

> **Regla general del notebook:** todo lo que aquí se decide se basa en cifras recalculadas en este mismo notebook (no se copian directamente los números del EDA), para que el pipeline sea reproducible de principio a fin desde el Excel crudo.


## 0. Carga de librerías y datos

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42
DATA_PATH = '../data/raw/accidentes-trafico.xlsx'

df = pd.read_excel(DATA_PATH)
n_original = len(df)
print(f'Dataset original: {df.shape[0]:,} filas | {df.shape[1]} columnas')

Dataset original: 199,078 filas | 26 columnas

## 1. Limpieza estructural (nivel persona)

Se aplican aquí las transformaciones básicas de higiene de texto **antes** de tomar ninguna decisión analítica:
- `strip()` de todas las columnas de texto.
- Homogeneización de un problema de encoding detectado en `Tramo Edad` (`"ANOS"` sin tilde conviviendo con `"AÑOS"` en la misma columna — si no se corrige, un `.map()` de tramos de edad pierde silenciosamente esas filas).
- Conversión de `FECHA` a `datetime` y extracción de `AÑO` / `MES`.
- Exclusión de `TESTIGO` (no es un implicado con lesividad propia).
- Eliminación de duplicados exactos — **recalculado tras excluir testigos**, no antes.


In [2]:
# --- Limpieza de texto ---
for col in df.select_dtypes('object').columns:
    df[col] = df[col].str.strip()

# --- Corrección de encoding en Tramo Edad ---
n_anos_sin_tilde = df['Tramo Edad'].str.contains('ANOS', na=False).sum()
df['Tramo Edad'] = df['Tramo Edad'].str.replace('ANOS', 'AÑOS', regex=False)
print(f'Filas corregidas por inconsistencia de encoding en Tramo Edad: {n_anos_sin_tilde:,}')

# --- Fechas ---
df['FECHA'] = pd.to_datetime(df['FECHA'])
df['AÑO'] = df['FECHA'].dt.year
df['MES'] = df['FECHA'].dt.month

Filas corregidas por inconsistencia de encoding en Tramo Edad: 23,785

In [3]:
# --- Exclusión de testigos ---
n_testigos = (df['TIPO PERSONA'] == 'TESTIGO').sum()
df = df[df['TIPO PERSONA'] != 'TESTIGO'].copy()
n_sin_testigos = len(df)

# --- Duplicados exactos (tras excluir testigos) ---
n_duplicados = df.duplicated().sum()
df = df.drop_duplicates().copy()
n_sin_duplicados = len(df)

print(f'Registros originales:            {n_original:>8,}')
print(f'Testigos excluidos:               {n_testigos:>8,}')
print(f'Tras excluir testigos:            {n_sin_testigos:>8,}')
print(f'Duplicados exactos eliminados:    {n_duplicados:>8,}')
print(f'Tras eliminar duplicados:         {n_sin_duplicados:>8,}')

Registros originales:             199,078
Testigos excluidos:                 24,725
Tras excluir testigos:             174,353
Duplicados exactos eliminados:       4,065
Tras eliminar duplicados:          170,288

**Nota:** el número de duplicados exactos depende del orden de operaciones. Sobre el Excel crudo (incluyendo testigos) hay 5.272 filas duplicadas; tras excluir testigos, el número correcto es el recalculado arriba. Es importante fijar y documentar el orden exacto de los pasos, porque cambia la cifra final.

In [4]:
# --- Nulos: valores textuales que representan "sin dato" ---
VALORES_NULOS = {'NO ASIGNADO', 'NO ASIGNADA', 'DESCONOCIDO', 'DESCONOCIDA'}
for col in df.select_dtypes('object').columns:
    df[col] = df[col].replace(VALORES_NULOS, pd.NA)

print('Nulos introducidos por columna (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))

Nulos introducidos por columna (top 10):
Tipo Vehiculo      11672
Tramo Edad          7971
LESIVIDAD           6435
SEXO                6228
Nº                  2618
LUGAR ACCIDENTE        0
Nº PARTE               0
CPFA Granizo           0
DISTRITO               0
FECHA                  0
dtype: int64

## 2. Construcción del target y agregación persona → accidente

Este es el paso más importante del preprocesado. El dataset crudo está a **nivel persona** (varias filas por accidente), pero el target definido en el EDA es la **lesividad máxima por accidente** (`Nº PARTE`). Hay que decidir explícitamente:

1. Qué hacer con las filas de `LESIVIDAD = NO ASIGNADA`.
2. Cómo agregar el resto de variables de nivel persona (sexo, edad, tipo de persona, tipo de vehículo) que no tienen un valor único por accidente.

**Decisión 2.1:** se eliminan las filas con `LESIVIDAD` nula **antes** de agregar. Se comprueba primero que ningún accidente se pierde por completo al hacerlo (es decir, que todo accidente conserva al menos una persona con lesividad conocida).


In [5]:
ORDEN_LESIVIDAD = ['IL', 'HL', 'HG', 'MT']

partes_todos = set(df['Nº PARTE'].unique())

df_pers = df[df['LESIVIDAD'].isin(ORDEN_LESIVIDAD)].copy()
df_pers['GRAVE'] = df_pers['LESIVIDAD'].isin(['HG', 'MT']).astype(int)

partes_con_dato = set(df_pers['Nº PARTE'].unique())
n_accidentes_perdidos = len(partes_todos - partes_con_dato)

print(f'Filas con LESIVIDAD no asignada eliminadas: {(len(df) - len(df_pers)):,}')
print(f'Personas tras el filtro:                    {len(df_pers):,}')
print(f'Accidentes únicos antes del filtro:          {len(partes_todos):,}')
print(f'Accidentes únicos tras el filtro:             {len(partes_con_dato):,}')
print(f'Accidentes perdidos por completo:             {n_accidentes_perdidos}')

Filas con LESIVIDAD no asignada eliminadas: 6,435
Personas tras el filtro:                    163,853
Accidentes únicos antes del filtro:          68,773
Accidentes únicos tras el filtro:             68,773
Accidentes perdidos por completo:             0

Ningún accidente se pierde por completo: todos conservan al menos una persona con lesividad conocida, así que el filtro es seguro.

**Decisión 2.2:** las tres variables de persona con mayor asociación con la gravedad según el EDA (`TIPO PERSONA`, `Tipo Vehiculo`, `Tramo Edad`) no pueden descartarse sin más al pasar a nivel accidente — perderíamos justo la señal más predictiva. Se agregan como **flags de implicación** (¿hubo al menos un peatón?, ¿una moto?, ¿un grupo de edad de riesgo?). Cualquier variable derivada más elaborada (interacciones, proporciones, edad media ponderada, etc.) se deja para el notebook de *feature engineering* — aquí solo se evita la pérdida de información básica.

In [6]:
GRUPOS_RIESGO_EDAD = [
    'DE 0 A 5 AÑOS', 'DE 6 A 9 AÑOS', 'DE 10 A 14 AÑOS', 'DE 15 A 17 AÑOS',
    'DE 65 A 69 AÑOS', 'DE 70 A 74 AÑOS', 'DE MAS DE 74 AÑOS'
]

agg_flags = df_pers.groupby('Nº PARTE').agg(
    GRAVE=('GRAVE', 'max'),
    N_PERSONAS_IMPLICADAS=('GRAVE', 'size'),
    INCLUYE_PEATON=('TIPO PERSONA', lambda s: (s == 'PEATON').any()),
    INCLUYE_MOTO=('Tipo Vehiculo', lambda s: s.isin(['MOTOCICLETA', 'CICLOMOTOR']).any()),
    INCLUYE_BICI=('Tipo Vehiculo', lambda s: (s == 'BICICLETA').any()),
    INCLUYE_EDAD_RIESGO=('Tramo Edad', lambda s: s.isin(GRUPOS_RIESGO_EDAD).any()),
).reset_index()

for c in ['INCLUYE_PEATON', 'INCLUYE_MOTO', 'INCLUYE_BICI', 'INCLUYE_EDAD_RIESGO']:
    agg_flags[c] = agg_flags[c].astype(int)

agg_flags.head()

     Nº PARTE  GRAVE  N_PERSONAS_IMPLICADAS  INCLUYE_PEATON  INCLUYE_MOTO  INCLUYE_BICI  INCLUYE_EDAD_RIESGO
0  2012/10000      0                      2               0             1             0                    0
1  2012/10002      0                      1               0             0             0                    1
2  2012/10003      0                      1               0             0             1                    0
3  2012/10005      0                      2               0             1             0                    0
4  2012/10007      0                      2               0             0             0                    0

**Decisión 2.3:** las columnas realmente invariantes por accidente (fecha, distrito, lugar, tipo de accidente, condiciones meteorológicas, estado del firme, nº de víctimas) se toman una única vez por `Nº PARTE`. Antes de darlo por hecho, se verifica que efectivamente no varían dentro de un mismo accidente (podría haber errores de captura).

In [7]:
COLS_ACCIDENTE = [
    'Nº PARTE', 'FECHA', 'AÑO', 'MES', 'RANGO HORARIO', 'DIA SEMANA', 'DISTRITO', 'LUGAR ACCIDENTE',
    'TIPO ACCIDENTE', 'Nº VICTIMAS *',
    'CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Seco', 'CPFA Nieve',
    'CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo', 'CPSV Seca Y Limpia'
]

# Verificación: ¿alguna de estas columnas varía dentro del mismo accidente?
inconsistencias = {c: (df.groupby('Nº PARTE')[c].nunique() > 1).sum() for c in COLS_ACCIDENTE if c != 'Nº PARTE'}
inconsistencias = {k: v for k, v in inconsistencias.items() if v > 0}
print('Columnas con valores inconsistentes dentro de un mismo accidente:', inconsistencias if inconsistencias else 'ninguna ✅')

df_accidente_info = df.drop_duplicates(subset='Nº PARTE')[COLS_ACCIDENTE]
df_final = df_accidente_info.merge(agg_flags, on='Nº PARTE', how='inner')

print(f'\nDataset a nivel accidente: {df_final.shape[0]:,} filas | {df_final.shape[1]} columnas')
print(f'Tasa de gravedad (GRAVE=1): {df_final["GRAVE"].mean()*100:.2f}%  (ratio ~{1/df_final["GRAVE"].mean():.0f}:1)')

Columnas con valores inconsistentes dentro de un mismo accidente: ninguna ✅

Dataset a nivel accidente: 68,773 filas | 28 columnas
Tasa de gravedad (GRAVE=1): 9.60%  (ratio ~10:1)

El resultado (68.773 accidentes, ~9.6% graves) coincide con la cifra de gravedad a nivel accidente reportada en el EDA (sección 5.2), lo que confirma que la agregación es coherente con el análisis previo.

## 3. Tipado y consistencia de variables

- Las binarias `CPFA_*` / `CPSV_*` pasan de `SI`/`NO` a `1`/`0` explícito.
- `RANGO HORARIO` (texto, 24 categorías) se convierte en `HORA` numérica (0-23) — más útil para el modelo y para crear tramos horarios en feature engineering.
- Se añade `ES_FINDE` a partir de `DIA SEMANA`.
- `LUGAR ACCIDENTE` es texto libre de altísima cardinalidad (miles de valores únicos) — no es utilizable tal cual como categórica. Se extrae `TIPO_VIA` (autovía / calle / avenida / paseo / plaza / glorieta / carretera / ronda / otros), **y además un flag `ES_CRUCE`**, ya que una parte muy relevante de los registros tiene formato `"CALLE 1 - CALLE 2"` (intersección de dos vías) y ese patrón se perdía por completo en la primera versión. Se conserva la columna original solo como referencia (no se usará en el modelo).


In [8]:
# --- Binarias CPFA / CPSV ---
COLS_BINARIAS = [c for c in COLS_ACCIDENTE if c.startswith('CPFA') or c.startswith('CPSV')]
for c in COLS_BINARIAS:
    assert set(df_final[c].unique()) <= {'SI', 'NO'}, f'{c} tiene valores inesperados'
    df_final[c] = (df_final[c] == 'SI').astype(int)

print('Binarias convertidas:', COLS_BINARIAS)

Binarias convertidas: ['CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Seco', 'CPFA Nieve', 'CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo', 'CPSV Seca Y Limpia']

In [9]:
# --- Hora numérica ---
df_final['HORA'] = df_final['RANGO HORARIO'].str.extract(r'DE\s+(\d{1,2}):')[0].astype(int)
assert df_final['HORA'].between(0, 23).all()

# --- Fin de semana ---
df_final['ES_FINDE'] = df_final['DIA SEMANA'].isin(['SABADO', 'DOMINGO']).astype(int)

print(df_final[['RANGO HORARIO', 'HORA', 'DIA SEMANA', 'ES_FINDE']].drop_duplicates(subset='RANGO HORARIO').sort_values('HORA').head())

        RANGO HORARIO  HORA DIA SEMANA  ES_FINDE
34   DE 00:00 A 00:59     0     MARTES         0
132    DE 1:00 A 1:59     1    DOMINGO         1
0      DE 2:00 A 2:59     2    DOMINGO         1
134    DE 3:00 A 3:59     3    DOMINGO         1
12     DE 4:00 A 4:59     4      LUNES         0

### 3.1 Detección de cruces en `LUGAR ACCIDENTE`

Muchos registros de `LUGAR ACCIDENTE` tienen el formato `"CALLE 1 - CALLE 2"`, que identifica un accidente en una **intersección** entre dos vías. Este patrón se pierde si se clasifica el texto en un único tipo de vía sin más — hay que extraerlo antes de nada.


In [ ]:
# --- ¿Cuántos accidentes son en realidad cruces? ---
n_valores_unicos = df_final['LUGAR ACCIDENTE'].nunique()
es_cruce_valores = df_final['LUGAR ACCIDENTE'].str.contains(' - ', na=False) #los guiones en casos como M-30 no llevan espacios antes y después
n_cruce_valores_unicos = df_final.loc[es_cruce_valores, 'LUGAR ACCIDENTE'].nunique()

print(f'Valores únicos de LUGAR ACCIDENTE:                {n_valores_unicos:,}')
print(f'  ...de los cuales tienen formato "calle - calle": {n_cruce_valores_unicos:,} ({n_cruce_valores_unicos/n_valores_unicos*100:.1f}%)')
print(f'Accidentes (filas) en cruce:                      {es_cruce_valores.sum():,} de {len(df_final):,} ({es_cruce_valores.mean()*100:.1f}%)')

Valores únicos de LUGAR ACCIDENTE:                13,958
  ...de los cuales tienen formato "calle - calle": 10,651 (76.3%)
Accidentes (filas) en cruce:                      31,927 de 68,773 (46.4%)

Casi la mitad de los accidentes (≈46%) ocurre en una intersección. Ignorar este patrón sería perder una variable potencialmente relevante, tal y como se apuntaba. Antes de asumir que es "decisivo" para la gravedad, se comprueba directamente sobre los datos:


In [11]:
df_final['ES_CRUCE'] = df_final['LUGAR ACCIDENTE'].str.contains(' - ', na=False).astype(int)

print('Tasa de gravedad según ES_CRUCE:')
print(df_final.groupby('ES_CRUCE')['GRAVE'].agg(['mean', 'count']).rename(columns={'mean': 'pct_grave'}))

Tasa de gravedad según ES_CRUCE:
          pct_grave  count
ES_CRUCE                  
0          0.094529  36846
1          0.097754  31927

**El efecto marginal de `ES_CRUCE` por sí solo es muy pequeño** (9.45% vs 9.78% de gravedad, apenas 0.3 puntos) — no es "decisivo" de forma aislada, como cabría esperar de la intuición inicial. No obstante, antes de descartarlo, se comprueba si el efecto está enmascarado por su relación con el tipo de vía y el tipo de accidente:


In [1]:
# Relación entre ES_CRUCE y TIPO ACCIDENTE (¿qué tipo de siniestro ocurre más en cruces?)
print('% de accidentes en cruce, por TIPO ACCIDENTE (ordenado):')
print((df_final.groupby('TIPO ACCIDENTE')['ES_CRUCE'].mean() * 100).round(1).sort_values(ascending=False))

% de accidentes en cruce, por TIPO ACCIDENTE (ordenado):


NameError: name 'df_final' is not defined

In [ ]:
# Efecto de ES_CRUCE DENTRO de cada TIPO ACCIDENTE (para no mezclar composiciones distintas)
tabla_tipo_acc = df_final.groupby(['TIPO ACCIDENTE', 'ES_CRUCE'])['GRAVE'].mean().unstack() * 100
tabla_tipo_acc.columns = ['pct_grave_sin_cruce', 'pct_grave_con_cruce']
tabla_tipo_acc['diferencia_pts'] = (tabla_tipo_acc['pct_grave_con_cruce'] - tabla_tipo_acc['pct_grave_sin_cruce']).round(2)
tabla_tipo_acc.round(2).sort_values('diferencia_pts', ascending=False)

**Corrección respecto a la primera versión:** en los dos tipos de accidente más frecuentes — colisión doble (55% del total) y atropello (15% del total), que juntos son el 70% del dataset — el cruce **aumenta** la tasa de gravedad (+1.2 puntos en ambos), igual que en colisión múltiple (+1.7) y vuelco (+4.2, aunque con pocos casos). Es decir, en las categorías que más pesan, el efecto va en la **misma** dirección que la intuición inicial, no en la contraria (como se afirmaba antes). En categorías menos frecuentes (caída de bicicleta, ciclomotor, choque contra objeto fijo) el signo se invierte, pero son grupos mucho más pequeños y más sensibles al ruido muestral.

El punto clave no es que el efecto sea uniforme — no lo es — sino que el efecto marginal (0.33 puntos) es mucho más pequeño que los efectos dentro de cada tipo (de 1 a 4 puntos en las categorías grandes) porque, al mezclar todos los tipos de accidente en una sola comparación, la distinta composición de tipos entre cruces y no-cruces (más colisión doble en los cruces, más atropello fuera de ellos) **diluye** ese efecto real, sin llegar a invertirlo del todo.

**Decisión:** se conserva `ES_CRUCE` como variable — no por su efecto marginal (débil y engañoso si se mira aislado), sino porque:
1. Es información estructural que no debe perderse en el preprocesado.
2. Su efecto real, condicionado al tipo de accidente y al tipo de vía, es más relevante que el efecto marginal — y ese es precisamente el tipo de variable de la que puede salir valor en feature engineering (p. ej. una interacción `TIPO_ACCIDENTE_x_CRUCE` o `TIPO_VIA_x_CRUCE`).
3. Descartarla ahora, basándose solo en el efecto marginal, sería un error: se estaría tirando una variable con señal real por mirarla de la forma equivocada.

Aquí está la explicación: los cruces están fuertemente asociados a **colisión doble** (~54% de los accidentes en cruce son de este tipo), que en el EDA (sección 10) ya se identificó como uno de los tipos de accidente de **menor** gravedad relativa, frente a atropellos o caídas de moto, que ocurren más a menudo en tramos rectos o vías de mayor velocidad. Es decir: los cruces no son irrelevantes, pero su efecto sobre la gravedad va **en dirección contraria** a la intuición inicial cuando se mira de forma aislada, porque está confundido con el tipo de accidente y el tipo de vía.

**Decisión:** se conserva `ES_CRUCE` como variable — no por su efecto marginal (débil), sino porque:
1. Es información estructural que no debe perderse en el preprocesado.
2. Su interacción con `TIPO_VIA` y `TIPO ACCIDENTE` sí muestra variación real (se ve más abajo), y esas interacciones son precisamente el tipo de variable que puede aportar valor en feature engineering.


In [13]:
# --- Tipo de vía a partir del primer tramo (antes del " - " si es cruce) ---
def tipo_via(texto):
    if pd.isna(texto):
        return np.nan
    primer_tramo = texto.split(' - ')[0].upper()
    if 'AUTOVIA' in primer_tramo or 'M-30' in primer_tramo or 'M-40' in primer_tramo:
        return 'AUTOVIA'
    if 'CALLE' in primer_tramo:
        return 'CALLE'
    if 'AVENIDA' in primer_tramo:
        return 'AVENIDA'
    if 'PASEO' in primer_tramo:
        return 'PASEO'
    if 'PLAZA' in primer_tramo:
        return 'PLAZA'
    if 'GLORIETA' in primer_tramo:
        return 'GLORIETA'
    if 'CARRETERA' in primer_tramo:
        return 'CARRETERA'
    if 'RONDA' in primer_tramo:
        return 'RONDA'
    return 'OTROS'

df_final['TIPO_VIA'] = df_final['LUGAR ACCIDENTE'].apply(tipo_via)
df_final['TIPO_VIA'].value_counts(dropna=False)

TIPO_VIA
CALLE        38927
AVENIDA      13931
PASEO         5322
AUTOVIA       4964
PLAZA         1859
OTROS         1620
GLORIETA      1018
CARRETERA      846
RONDA          286

In [14]:
# Tasa de gravedad cruzando TIPO_VIA x ES_CRUCE (para ver dónde sí importa el cruce)
tabla = df_final.groupby(['TIPO_VIA', 'ES_CRUCE'])['GRAVE'].agg(['mean', 'count']).rename(columns={'mean': 'pct_grave'})
tabla['pct_grave'] = (tabla['pct_grave'] * 100).round(2)
tabla

                    pct_grave  count
TIPO_VIA  ES_CRUCE                  
AUTOVIA   0              6.98   3984
          1              6.22    980
AVENIDA   0              9.57   6356
          1              9.90   7575
CALLE     0              9.83  18442
          1              9.87  20485
CARRETERA 0              9.96    683
          1             12.88    163
GLORIETA  0              6.78    634
          1             10.94    384
OTROS     0             12.60   1238
          1              9.16    382
PASEO     0              9.56   3839
          1              9.37   1483
PLAZA     0              8.36   1435
          1             10.61    424
RONDA     0             13.19    235
          1             13.73     51

En vías de tipo `GLORIETA` y `CARRETERA` el cruce sí eleva bastante la tasa de gravedad (p. ej. glorieta: 6.8% sin cruce vs 10.9% con cruce), mientras que en `CALLE` o `AVENIDA` apenas cambia. Esto confirma que `ES_CRUCE` **no es una variable de efecto único**, sino que su valor está en la interacción con el tipo de vía — justo el tipo de patrón que interesa capturar de cara al feature engineering (ej. una variable combinada `TIPO_VIA_x_CRUCE`).

> **Nota para feature engineering:** la clasificación de `TIPO_VIA` mediante reglas de texto es una primera aproximación razonable, pero es mejorable (p. ej. geocodificar `LUGAR ACCIDENTE` con `osmnx`/Nominatim para obtener coordenadas y cruzar con capas de velocidad de vía, como ya se explora en el EDA geográfico). Aquí se deja resuelto lo mínimo necesario para no descartar la columna de raíz ni el patrón de cruce.

## 4. Missing data

Se recalculan los nulos **sobre el dataset ya a nivel accidente**, no sobre el crudo (los porcentajes cambian tras la agregación).


In [15]:
missing = df_final.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df_final) * 100).round(2)
pd.DataFrame({'n_nulos': missing, 'pct': missing_pct})

Empty DataFrame
Columns: [n_nulos, pct]
Index: []

No quedan nulos relevantes a nivel accidente en las variables que se van a modelizar (las únicas columnas con nulos, si las hay, son residuales como `TIPO_VIA` u otras de texto libre que no se usarán directamente en el modelo). Regla aplicada, para dejar constancia por si el pipeline se re-ejecuta con datos nuevos que sí tengan huecos:

- **Categóricas:** imputar con la categoría explícita `"Desconocido"` (no eliminar filas) — coherente con que, en este dominio, el propio "no asignado" resultó ser informativo en el EDA (p. ej. en `SEXO` o `Tipo Vehiculo`).
- **Numéricas:** imputar con la mediana **calculada solo sobre el conjunto de entrenamiento** (ver Fase 6), nunca sobre el dataset completo, para evitar fuga de información del test.


In [16]:
COLS_CATEGORICAS_MODELO = ['DISTRITO', 'TIPO ACCIDENTE', 'TIPO_VIA', 'DIA SEMANA']
for c in COLS_CATEGORICAS_MODELO:
    df_final[c] = df_final[c].astype('object').fillna('Desconocido')

print('Nulos restantes en categóricas de modelo:', df_final[COLS_CATEGORICAS_MODELO].isna().sum().sum())

Nulos restantes en categóricas de modelo: 0

## 5. Variable numérica y outliers

`Nº VICTIMAS *` presenta una distribución muy asimétrica (la mayoría de accidentes tienen 1 víctima, con una cola larga de casos con varias). Antes de transformar, se revisan los valores extremos para descartar que sean errores de captura.


In [17]:
print(df_final['Nº VICTIMAS *'].describe())
print()
print('Accidentes con 9 o más víctimas:')
print(df_final['Nº VICTIMAS *'].value_counts().sort_index().loc[lambda s: s.index >= 9])

count    68773.000000
mean         1.279092
std          0.721309
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         19.000000
Name: Nº VICTIMAS *, dtype: float64

Accidentes con 9 o más víctimas:
Nº VICTIMAS *
9     9
10    8
11    3
12    2
13    2
14    2
15    1
16    1
18    1
19    2
Name: count, dtype: int64

Los valores altos (hasta 19 víctimas) son poco frecuentes pero plausibles en accidentes reales de tráfico urbano con varios vehículos implicados (p. ej. colisiones múltiples o atropellos con varios heridos) — no se detectan valores imposibles (negativos, cero, o desproporcionados sin explicación), así que **no se eliminan como outliers**, se tratan como cola larga real de la distribución. Se añade una versión transformada en logaritmo, quedando ambas versiones disponibles para que feature engineering / modelización decida cuál usar.

In [18]:
df_final['N_VICTIMAS_LOG'] = np.log1p(df_final['Nº VICTIMAS *'])
df_final[['Nº VICTIMAS *', 'N_VICTIMAS_LOG']].describe()

       Nº VICTIMAS *  N_VICTIMAS_LOG
count   68773.000000    68773.000000
mean        1.279092        0.792485
std         0.721309        0.225994
min         1.000000        0.693147
25%         1.000000        0.693147
50%         1.000000        0.693147
75%         1.000000        0.693147
max        19.000000        2.995732

## 6. Train / test split

Se separa el dataset **antes** de cualquier paso que dependa de estadísticos calculados sobre los datos (medias, medianas, encoders de frecuencia, etc.), para evitar fuga de información.

Al tratarse de una serie de 7 años (2012-2018), se valoran dos estrategias:

1. **Split temporal** (train: 2012-2017, test: 2018): más realista si el modelo se plantea como una herramienta que se entrenaría con histórico para predecir accidentes futuros. Tiene la ventaja añadida de que permite comprobar si el modelo generaliza a un año que no ha visto, con la tendencia temporal real (se observa más abajo que la tasa de gravedad ha ido bajando año a año).
2. **Split aleatorio estratificado por `GRAVE`**: más estándar y con distribución de clases idéntica en train y test, útil si el objetivo del TFM es puramente comparar modelos sobre la misma distribución.

Se generan **ambos** y se deja documentada la recomendación: usar el split temporal como validación principal (más honesto), y el estratificado como referencia secundaria para comparar métricas con la literatura o entre modelos.


In [19]:
print('Tasa de gravedad por año:')
print((df_final.groupby('AÑO')['GRAVE'].mean() * 100).round(2))

Tasa de gravedad por año:
AÑO
2012    10.23
2013    10.66
2014    10.87
2015     9.72
2016     9.13
2017     8.77
2018     8.19
Name: GRAVE, dtype: float64

In [20]:
# --- Split temporal ---
train_temporal = df_final[df_final['AÑO'] <= 2017].copy()
test_temporal = df_final[df_final['AÑO'] == 2018].copy()

print('SPLIT TEMPORAL')
print(f'  Train (2012-2017): {train_temporal.shape[0]:,} filas | {train_temporal["GRAVE"].mean()*100:.2f}% graves')
print(f'  Test  (2018):      {test_temporal.shape[0]:,} filas | {test_temporal["GRAVE"].mean()*100:.2f}% graves')

SPLIT TEMPORAL
  Train (2012-2017): 58,095 filas | 9.86% graves
  Test  (2018):      10,678 filas | 8.19% graves

In [21]:
# --- Split aleatorio estratificado (referencia secundaria) ---
train_rand, test_rand = train_test_split(
    df_final, test_size=0.2, stratify=df_final['GRAVE'], random_state=RANDOM_STATE
)

print('SPLIT ALEATORIO ESTRATIFICADO')
print(f'  Train: {train_rand.shape[0]:,} filas | {train_rand["GRAVE"].mean()*100:.2f}% graves')
print(f'  Test:  {test_rand.shape[0]:,} filas | {test_rand["GRAVE"].mean()*100:.2f}% graves')

SPLIT ALEATORIO ESTRATIFICADO
  Train: 55,018 filas | 9.60% graves
  Test:  13,755 filas | 9.60% graves

**Importante:** cualquier imputación con mediana, encoding de frecuencia/target-encoding u otra transformación que "aprenda" de los datos debe **ajustarse solo con `train_temporal` (o `train_rand`)** y aplicarse después a test — eso ya corresponde al notebook de feature engineering, pero se deja aquí la advertencia porque es el error de fuga de información más habitual en este tipo de proyecto.

Sobre el desbalanceo de clases (~9.6% graves): **no se corrige en el preprocessing** (ni con SMOTE ni con submuestreo). Es una decisión de modelización, no de preprocesado — se aplica, si procede, únicamente sobre `train`, nunca sobre `test`, tal y como ya se apuntaba en las conclusiones del EDA.

## 7. Control de calidad final y guardado

Tabla resumen de todo el proceso, de principio a fin, y guardado de los artefactos para el siguiente notebook.


In [22]:
resumen = pd.DataFrame({
    'Paso': [
        'Registros originales (nivel persona)',
        'Tras excluir TESTIGO',
        'Tras eliminar duplicados exactos',
        'Tras eliminar LESIVIDAD no asignada',
        'Accidentes únicos (nivel accidente, dataset final)',
    ],
    'Filas': [
        n_original,
        n_sin_testigos,
        n_sin_duplicados,
        len(df_pers),
        len(df_final),
    ]
})
resumen

                                                 Paso   Filas
0                Registros originales (nivel persona)  199078
1                                Tras excluir TESTIGO  174353
2                    Tras eliminar duplicados exactos  170288
3                 Tras eliminar LESIVIDAD no asignada  163853
4  Accidentes únicos (nivel accidente, dataset final)   68773

In [23]:
print('Dimensiones finales:', df_final.shape)
print('Columnas:', list(df_final.columns))
print()
print('Duplicados en Nº PARTE (debe ser 0):', df_final['Nº PARTE'].duplicated().sum())
print('Nulos en la variable target GRAVE (debe ser 0):', df_final['GRAVE'].isna().sum())

Dimensiones finales: (68773, 33)
Columnas: ['Nº PARTE', 'FECHA', 'AÑO', 'MES', 'RANGO HORARIO', 'DIA SEMANA', 'DISTRITO', 'LUGAR ACCIDENTE', 'TIPO ACCIDENTE', 'Nº VICTIMAS *', 'CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Seco', 'CPFA Nieve', 'CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo', 'CPSV Seca Y Limpia', 'GRAVE', 'N_PERSONAS_IMPLICADAS', 'INCLUYE_PEATON', 'INCLUYE_MOTO', 'INCLUYE_BICI', 'INCLUYE_EDAD_RIESGO', 'HORA', 'ES_FINDE', 'ES_CRUCE', 'TIPO_VIA', 'N_VICTIMAS_LOG']

Duplicados en Nº PARTE (debe ser 0): 0
Nulos en la variable target GRAVE (debe ser 0): 0

In [24]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_final.to_csv('../data/processed/accidentes_accidente_nivel_clean.csv', index=False)
train_temporal.to_csv('../data/processed/train_temporal.csv', index=False)
test_temporal.to_csv('../data/processed/test_temporal.csv', index=False)
train_rand.to_csv('../data/processed/train_random.csv', index=False)
test_rand.to_csv('../data/processed/test_random.csv', index=False)

# Dataset a nivel persona ya limpio, por si feature engineering necesita
# construir variables más finas de agregación (proporciones, edad media, etc.)
df_pers.to_csv('../data/processed/personas_clean.csv', index=False)

print('Ficheros guardados en ../data/processed/')

Ficheros guardados en ../data/processed/

## Resumen de decisiones tomadas en este notebook

| # | Decisión |
|---|----------|
| 1 | Se corrige una inconsistencia de encoding en `Tramo Edad` (`ANOS` → `AÑOS`) antes de cualquier mapeo de categorías |
| 2 | Duplicados exactos se calculan **tras** excluir testigos, no antes (el número cambia según el orden) |
| 3 | El target (`GRAVE`) se construye a **nivel accidente** como máximo de la gravedad de las personas implicadas, verificando antes que ningún accidente se pierde por completo al filtrar `LESIVIDAD` nula |
| 4 | Las variables de persona más predictivas según el EDA (tipo de persona, tipo de vehículo, tramo de edad) se preservan como *flags* de implicación a nivel accidente, en vez de perderse en la agregación |
| 5 | Se verifica explícitamente que las columnas "de accidente" (fecha, distrito, meteorología, etc.) son realmente invariantes dentro de cada `Nº PARTE` antes de asumirlo |
| 6 | `RANGO HORARIO` y `LUGAR ACCIDENTE` (texto libre de alta cardinalidad) se transforman en variables utilizables (`HORA` numérica, `TIPO_VIA`) |
| 7 | Se detecta y conserva el patrón de **intersección** en `LUGAR ACCIDENTE` (`ES_CRUCE`, ~46% de los accidentes); su efecto marginal es débil pero muestra interacción real con `TIPO_VIA` y `TIPO ACCIDENTE`, así que se preserva para feature engineering en vez de descartarlo |
| 8 | Los nulos residuales en categóricas se imputan como categoría explícita `"Desconocido"`, no se eliminan filas |
| 9 | `Nº VICTIMAS *` no se trata como outlier (los valores altos son accidentes reales con varios implicados); se añade versión log-transformada sin eliminar la original |
| 10 | El split train/test se hace **antes** de cualquier imputación estadística o encoding, con dos estrategias documentadas (temporal y aleatorio estratificado) |
| 11 | El desbalanceo de clases se deja explícitamente para la fase de modelización, no se corrige aquí |

**Siguiente paso:** `03_Feature_Engineering.ipynb`, partiendo de `train_temporal.csv` / `test_temporal.csv` (o sus equivalentes aleatorios) ya guardados.
